In [ ]:
import pandas as pd
import pandas as pd
import utils
import os
from tqdm import tqdm
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, accuracy_score
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder, TargetEncoder, RobustScaler, TargetEncoder
from sklearn.pipeline import Pipeline

%load_ext autoreload
%autoreload 2

Basic Model - 30% accuracy achieved

In [ ]:
data_folder = 'data/'
raw_train_df = pd.read_csv(data_folder + 'train.csv')
raw_test_df = pd.read_csv(data_folder + 'test.csv')
train_demo_df = pd.read_csv(data_folder + 'train_demographics.csv')
test_demo_df = pd.read_csv(data_folder + 'test_demographics.csv')
temp_calculations_folder_name = 'temp_calculations/'
os.makedirs(temp_calculations_folder_name, exist_ok=True)

feat_columns = ['sequence_type', 'sequence_id', 'sequence_counter', 'subject', 'orientation', 'behavior', 'phase', 'gesture']
acc_columns = ['acc_x', 'acc_y', 'acc_z']
rot_columns = ['rot_x', 'rot_y', 'rot_z']
temp_columns = ['thm_1', 'thm_2', 'thm_3', 'thm_4', 'thm_5']
tof_columns = raw_train_df.columns[raw_train_df.columns.str.startswith('tof')]
non_device_cols = ['sequence_type', 'sequence_id', 'sequence_counter', 'subject', 'orientation', 'behavior', 'phase', 'gesture', 'adult_child', 'age', 'sex', 'handedness', 'height_cm', 'shoulder_to_wrist_cm', 'elbow_to_wrist_cm']
sampling_rate = 100
exp_name = 'imu_only_simple'

In [ ]:
train_df = raw_train_df.set_index('row_id')
train_df.head(1)

In [ ]:
# Your variables
exp_name = 'fft_v1_baseline' # Named for your experiment
sampling_rate = 100
phase_1_cols = ['acc_x', 'acc_y', 'acc_z', 'sequence_counter', 'phase', 'behavior', 'orientation', 'subject', 'gesture', 'sequence_id']
subjects_list = train_df['subject'].unique()

# 1. Create the folder named after your variable
output_dir = f'temp_calculations/{exp_name}'
os.makedirs(output_dir, exist_ok=True)

subject_set_list = []

for single_subject in tqdm(subjects_list, desc='Subject', position=0, leave=True):
    
    # 2. Define the path for this specific subject
    subject_file = f"{output_dir}/{single_subject}.csv"
    
    # 3. CHECK: If file exists, load it and skip the calculation
    if os.path.exists(subject_file):
        temp_processed_subject_df = pd.read_csv(subject_file, index_col=0)
        subject_set_list.append(temp_processed_subject_df)
        continue

    # --- START YOUR ORIGINAL LOGIC ---
    single_subject_df = train_df.loc[train_df['subject'] == single_subject, phase_1_cols]

    sequence_set_list = []
    for single_sequence in tqdm(single_subject_df['sequence_id'].unique(), desc='Sequence', position=1, leave=False):
        single_gesture_df = single_subject_df.loc[single_subject_df['sequence_id'] == single_sequence]

        temp_orientation = single_gesture_df['orientation'].unique()[0]
        temp_subject = single_gesture_df['subject'].unique()[0]
        temp_gesture = single_gesture_df['gesture'].unique()[0]

        # Get Acceleration
        temp_acc_time_df = single_gesture_df[['acc_x', 'acc_y', 'acc_z']]
        temp_acc_fft_df = temp_acc_time_df.apply(utils.convert_frame_to_fft, axis=0, args=(sampling_rate,))
        
        # Zeroing DC Offset
        temp_acc_fft_df.loc[0:1] = 0

        features_acc = (
            temp_acc_fft_df
            .apply(lambda col: utils.extract_features(col, sampling_rate=sampling_rate))
            .T
            .add_prefix("")
        )

        single_row_df = features_acc.stack()

        single_row_df.index = [f"{axis}_{feat}" for axis, feat in single_row_df.index]
        single_row_df = single_row_df.to_frame().T
        single_row_df.index = [single_sequence]

        single_row_df.loc[single_sequence, 'orientation'] = temp_orientation
        single_row_df.loc[single_sequence, 'subject'] = temp_subject
        single_row_df.loc[single_sequence, 'gesture'] = temp_gesture

        sequence_set_list.append(single_row_df)

    # Combine the sequences for this subject
    temp_processed_subject_df = pd.concat(sequence_set_list)
    
    # 4. SAVE: Save the subject-level CSV to your experiment folder
    temp_processed_subject_df.to_csv(subject_file)
    
    subject_set_list.append(temp_processed_subject_df)
    # --- END YOUR ORIGINAL LOGIC ---

# Final Merge
feature_df = pd.concat(subject_set_list).merge(train_demo_df, on='subject', how='left')
feature_df.head(1)

In [ ]:
corr_matrix = feature_df[feature_df.columns.difference(['orientation', 'subject', 'gesture'])].corr()

# Plot the heatmap
plt.figure(figsize=(12, 8))
sns.heatmap(corr_matrix, annot=True, cmap='coolwarm', fmt=".2f", linewidths=0.5)
plt.title('Vibration Feature Correlation Heatmap')
plt.show()

In [ ]:
categorical_cols = ['orientation']
acc_cols = feature_df.columns[feature_df.columns.str.startswith('acc')].tolist()
demo_cols = train_demo_df.columns[~train_demo_df.columns.str.contains('subject')].tolist()
numerical_cols = acc_cols + demo_cols

# Preprocessor
preprocessor = ColumnTransformer(
    transformers=[
        ('cat', TargetEncoder(), categorical_cols),
        ('num', 'passthrough', numerical_cols),
        ('drop', 'drop', ['subject'])
    ]
)

# Create pipeline
pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', RandomForestClassifier(n_estimators=100, random_state=42))
])

# Split data
X = feature_df.drop('gesture', axis=1)
y = feature_df['gesture']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Fit pipeline
pipeline.fit(X_train, y_train)

# Predict and evaluate
y_pred = pipeline.predict(X_test)
print(f"Accuracy: {accuracy_score(y_test, y_pred):.4f}")
print("\nClassification Report:\n", classification_report(y_test, y_pred))

In [ ]:
X_train